In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import drive
drive.mount('/content/drive')

# Load the dataset

!cp "/content/drive/MyDrive/hasil_data_gabungan.csv" "./data.csv"
df = pd.read_csv("data.csv")

# Clean and convert data to numeric
for col in ["Kasus Malaria", "Curah Hujan", "Kelembapan", "Suhu"]:
    df[col] = df[col].astype(str).str.replace(",", "").str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(inplace=True)

# Create time series sequences
def create_sequences(data, n_steps=2):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps, 1:])  # Features
        y.append(data[i+n_steps, 0])     # Target
    return np.array(X), np.array(y)

# Inverse transform function
def inverse_scale(preds, scaler):
    dummy = np.zeros((len(preds), 3))
    return scaler.inverse_transform(np.hstack((preds, dummy)))[:, 0]

training_histories = {}
# Model configuration
n_steps = 2
performance_results = []
provinces = df["Nama Provinsi"].unique()

for province in provinces:
    print(f"\nProcessing: {province}")
    df_prov = df[df["Nama Provinsi"] == province].sort_values("Tahun").copy()

    if len(df_prov) < 5:
        print(f"Skipping {province} due to insufficient data.")
        continue

    # Normalize data
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df_prov[["Kasus Malaria", "Curah Hujan", "Kelembapan", "Suhu"]])

    X, y = create_sequences(scaled, n_steps)
    if len(X) < 1:
        print(f"Skipping {province} due to insufficient sequences.")
        continue

    # Split data
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # Define 5-layer LSTM model
    model = Sequential()
    model.add(LSTM(8, return_sequences=True, input_shape=(n_steps, X.shape[2])))
    model.add(Dropout(0.2))
    model.add(LSTM(24, return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(16, return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(16, return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(8))  # Final LSTM without return_sequences
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='linear'))

    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

    # Fit model
    early_stop = EarlyStopping(monitor='val_loss', patience=550, restore_best_weights=True)
    history = model.fit(X_train, y_train, epochs=4000, batch_size=1, verbose=0,
              callbacks=[early_stop], validation_data=(X_test, y_test))
    training_histories[province] = history.history
    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    y_train_inv = inverse_scale(y_train.reshape(-1,1), scaler)
    y_train_pred_inv = inverse_scale(y_train_pred, scaler)
    y_test_inv = inverse_scale(y_test.reshape(-1,1), scaler)
    y_test_pred_inv = inverse_scale(y_test_pred, scaler)

    # Metrics
    train_rmse = math.sqrt(mean_squared_error(y_train_inv, y_train_pred_inv))
    train_mae = mean_absolute_error(y_train_inv, y_train_pred_inv)
    test_rmse = math.sqrt(mean_squared_error(y_test_inv, y_test_pred_inv))
    test_mae = mean_absolute_error(y_test_inv, y_test_pred_inv)
    train_r2 = r2_score(y_train_inv, y_train_pred_inv)
    test_r2 = r2_score(y_test_inv, y_test_pred_inv)

    performance_results.append({
        'Province': province,
        'Model': 'LSTM_5Layers',
        'Train RMSE': train_rmse,
        'Train MAE': train_mae,
        'Train R2': train_r2,
        'Test RMSE': test_rmse,
        'Test MAE': test_mae,
        'Test R2': test_r2
    })

# Save results
performance_df = pd.DataFrame(performance_results)
performance_df.to_csv("performance_results_per_province.csv", index=False)
print("\nPerformance Summary:")
print(performance_df)

import matplotlib.pyplot as plt
import os

# Pastikan folder penyimpanan ada
drive_output_dir = "/content/drive/MyDrive/DATA/plots_loss_malaria_5layer_iklim lokal"
os.makedirs(drive_output_dir, exist_ok=True)

# Loop untuk setiap provinsi yang ada di training_histories
for prov, hist in training_histories.items():
    plt.figure(figsize=(6, 4))
    plt.plot(hist['loss'], label='Train')
    plt.plot(hist['val_loss'], label='Validation')
    plt.title(f"Model loss - {prov}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss (MSE)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    # Buat nama file yang rapi
    filename = prov.replace(" ", "_").replace("/", "_")
    path = os.path.join(drive_output_dir, f"loss_5 layer_malaria & iklim lokal_{filename}.png")
    plt.savefig(path)
    plt.close()

print(f"✅ Semua grafik model loss disimpan ke: {drive_output_dir}")

Mounted at /content/drive

Processing: ACEH


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 854ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 801ms/step

Processing: SUMATERA UTARA


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 798ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 977ms/step

Processing: SUMATERA BARAT


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 723ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 782ms/step

Processing: PAPUA BARAT


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 750ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step

Processing: PAPUA


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 826ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 912ms/step

Performance Summary:
                Province         Model    Train RMSE     Train MAE  Train R2  \
0                   ACEH  LSTM_5Layers    123.284823    104.002886  0.972536   
1         SUMATERA UTARA  LSTM_5Layers   3602.542314   2916.914401  0.548770   
2         SUMATERA BARAT  LSTM_5Layers    685.634665    578.670420 -2.307394   
3                   RIAU  LSTM_5Layers    556.999126    485.400432  0.081526   
4                  JAMBI  LSTM_5Layers   2026.888603   1341.804957 -0.678321   
5       SUMATERA SELATAN  LSTM_5Layers   1768.302463   1473.731976 -2.207957   
6               BENGKULU  LSTM_5Layers   4294.554845   3017.651660 -0.922310   
7                LAMPUNG  LSTM_5Layers   2409.451264   2089.900150 -2.288198   
8   KEP. BANGKA BELITUNG  LSTM_5Layers   1308.779833    810.106873 -0.566913   
9              KEP. RIAU  LSTM_5Layers   1480.310038    845.464388 -0.484023   
10           DKI JAK

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import drive
drive.mount('/content/drive')

# Load the dataset

!cp "/content/drive/MyDrive/hasil_data_gabungan.csv" "./data.csv"
df = pd.read_csv("data.csv")

# Clean and convert data to numeric
for col in ["Kasus Malaria", "Curah Hujan", "Kelembapan", "Suhu"]:
    df[col] = df[col].astype(str).str.replace(",", "").str.strip()
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(inplace=True)

# Create time series sequences
def create_sequences(data, n_steps=2):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps, 1:])  # Features
        y.append(data[i+n_steps, 0])     # Target
    return np.array(X), np.array(y)

# Inverse transform function
def inverse_scale(preds, scaler):
    dummy = np.zeros((len(preds), 3))
    return scaler.inverse_transform(np.hstack((preds, dummy)))[:, 0]

training_histories = {}
# Model configuration
n_steps = 2
performance_results = []
provinces = df["Nama Provinsi"].unique()

for province in provinces:
  if province  == "PAPUA":
    print(f"\nProcessing: {province}")
    df_prov = df[df["Nama Provinsi"] == province].sort_values("Tahun").copy()

    if len(df_prov) < 5:
        print(f"Skipping {province} due to insufficient data.")
        continue

    # Normalize data
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df_prov[["Kasus Malaria", "Curah Hujan", "Kelembapan", "Suhu"]])

    X, y = create_sequences(scaled, n_steps)
    if len(X) < 1:
        print(f"Skipping {province} due to insufficient sequences.")
        continue

    # Split data
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # Define 5-layer LSTM model
    model = Sequential()
    model.add(LSTM(8, return_sequences=True, input_shape=(n_steps, X.shape[2])))
    model.add(Dropout(0.2))
    model.add(LSTM(24, return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(16, return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(16, return_sequences=True))
    model.add(Dropout(0.2))
    model.add(LSTM(8))  # Final LSTM without return_sequences
    model.add(Dropout(0.2))
    model.add(Dense(1, activation='linear'))

    model.compile(optimizer=Adam(learning_rate=0.5), loss='mse')

    # Fit model
    early_stop = EarlyStopping(monitor='val_loss', patience=500, restore_best_weights=True)
    history = model.fit(X_train, y_train, epochs=4000, batch_size=1, verbose=0,
              callbacks=[early_stop], validation_data=(X_test, y_test))
    training_histories[province] = history.history
    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    y_train_inv = inverse_scale(y_train.reshape(-1,1), scaler)
    y_train_pred_inv = inverse_scale(y_train_pred, scaler)
    y_test_inv = inverse_scale(y_test.reshape(-1,1), scaler)
    y_test_pred_inv = inverse_scale(y_test_pred, scaler)

    # Metrics
    train_rmse = math.sqrt(mean_squared_error(y_train_inv, y_train_pred_inv))
    train_mae = mean_absolute_error(y_train_inv, y_train_pred_inv)
    test_rmse = math.sqrt(mean_squared_error(y_test_inv, y_test_pred_inv))
    test_mae = mean_absolute_error(y_test_inv, y_test_pred_inv)
    train_r2 = r2_score(y_train_inv, y_train_pred_inv)
    test_r2 = r2_score(y_test_inv, y_test_pred_inv)

    performance_results.append({
        'Province': province,
        'Model': 'LSTM_5Layers',
        'Train RMSE': train_rmse,
        'Train MAE': train_mae,
        'Train R2': train_r2,
        'Test RMSE': test_rmse,
        'Test MAE': test_mae,
        'Test R2': test_r2
    })

# Save results
performance_df = pd.DataFrame(performance_results)
performance_df.to_csv("performance_results_per_province.csv", index=False)
print("\nPerformance Summary:")
print(performance_df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Processing: PAPUA


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 652ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step

Performance Summary:
  Province         Model     Train RMSE      Train MAE   Train R2  \
0    PAPUA  LSTM_5Layers  142025.372743  135949.341867 -10.942812   

      Test RMSE      Test MAE   Test R2  
0  61998.719446  54535.626785 -0.625753  
